# QSAR Regressor Comparison (Clean Version)
This notebook compares regression models using LazyRegressor and exports **only the corrected CSV outputs**.
Old CSV generation has been removed.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from lazypredict.Supervised import LazyRegressor


## Load Dataset

In [ ]:
# Load dataset
df = pd.read_csv('your_dataset.csv')

# Define features and target
X = df.drop('activity', axis=1)
Y = df['activity']


## Train/Test Split

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

print('Train size:', X_train.shape)
print('Test size:', X_test.shape)


## Run Model Comparison

In [ ]:
clf = LazyRegressor(verbose=0, ignore_warnings=True)

models, predictions = clf.fit(X_train, X_test, Y_train, Y_test)


## Test Performance

In [ ]:
# Extract test metrics
test_performance = models[['R-Squared','RMSE']]

# Sort by best models
test_performance = test_performance.sort_values(by='R-Squared', ascending=False)

# Save CSV
test_performance.to_csv('performance_test_set_20_percent.csv')

test_performance.head()


## Training Performance

In [ ]:
train_results = {}

for name, model in clf.models.items():
    y_pred_train = model.predict(X_train)

    r2 = r2_score(Y_train, y_pred_train)
    rmse = np.sqrt(mean_squared_error(Y_train, y_pred_train))

    train_results[name] = {
        'R-Squared': r2,
        'RMSE': rmse
    }

train_performance = pd.DataFrame(train_results).T

# Sort by performance
train_performance = train_performance.sort_values(by='R-Squared', ascending=False)

# Save CSV
train_performance.to_csv('performance_train_set_80_percent.csv')

train_performance.head()


## Combined QSAR Model Comparison Table

In [ ]:
combined = train_performance.join(test_performance, lsuffix='_train', rsuffix='_test')

combined = combined.sort_values(by='R-Squared_test', ascending=False)

combined.to_csv('qsar_model_comparison.csv')

combined.head()
